# 01 · Backpropagation, pérdida y optimizadores


Ejecuta las celdas en orden. Antes de mirar la solución, intenta la actividad y anota una predicción. Si cambias parámetros, vuelve a ejecutar desde la celda de creación del modelo: llamar a `fit` otra vez continúa el entrenamiento anterior.

**Entorno:** Google Colab o Jupyter con `pip install tensorflow numpy matplotlib scikit-learn notebook`. Reinicia el kernel tras instalar si es necesario. No se necesitan descargas de conjuntos de datos. Los resultados y tiempos pueden variar por versión, CPU y aleatoriedad.

**Tiempo:** 35–40 minutos.

## 1. Una neurona, de principio a fin
Para un ejemplo, $\hat y=wx+b$ y $L=(\hat y-y)^2$. La pasada hacia adelante calcula la predicción y la pérdida. La retropropagación aplica la regla de la cadena para obtener gradientes:

$\frac{\partial L}{\partial w}=2(\hat y-y)x$, $\frac{\partial L}{\partial b}=2(\hat y-y)$.

Con SGD, $w \leftarrow w-\eta\frac{\partial L}{\partial w}$; $\eta$ es la tasa de aprendizaje. Un lote promedia la pérdida de sus ejemplos. El optimizador usa los gradientes: SGD aplica el paso; Adam adapta el tamaño de los pasos a partir de estimaciones acumuladas. La pérdida mide el objetivo de entrenamiento; una métrica, como MAE, ayuda a interpretarlo. En redes más profundas la regla de la cadena se aplica capa por capa.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)

## 2. Comprueba la regla de la cadena

Con $x=2$, $y=5$, $w=1$ y $b=0$, predice la pérdida y ambos gradientes. ¿En qué dirección debería cambiar cada parámetro?

In [ ]:
x = tf.constant([[2.0]], dtype=tf.float32)
y = tf.constant([[5.0]], dtype=tf.float32)
w = tf.Variable([[1.0]], dtype=tf.float32)
b = tf.Variable([0.0], dtype=tf.float32)
with tf.GradientTape() as tape:
    pred = x @ w + b
    loss = tf.reduce_mean(tf.square(pred - y))
dw, db = tape.gradient(loss, [w, b])
print("Predicción:", pred.numpy().item(), "Pérdida:", loss.numpy().item())
print("Gradientes w y b:", dw.numpy().item(), db.numpy().item())

La predicción es 2, la pérdida 9 y los gradientes son −12 y −6. Para una tasa positiva, restar estos gradientes incrementa ambos parámetros.

## 3. Entrena y observa los pasos

Compara un paso manual con el paso del optimizador. Reinicia las variables cuando repitas el experimento.

In [ ]:
eta = 0.1
print("Paso manual:", (w - eta * dw).numpy().item(), (b - eta * db).numpy().item())
opt = keras.optimizers.SGD(learning_rate=eta)
opt.apply_gradients([(dw, w), (db, b)])
print("Paso de SGD:", w.numpy().item(), b.numpy().item())

## 4. Tu turno: compara optimizadores

Con los mismos datos sintéticos de una recta, inicialización, lotes y épocas, compara SGD y Adam. Grafica entrenamiento y validación y comenta si una pérdida menor al final demuestra que un optimizador es siempre mejor. Empieza con tasas de 0,03.

In [ ]:
rng = np.random.default_rng(42)
X = rng.uniform(-2, 2, (400, 1)).astype("float32")
y_data = (3 * X[:, 0] + 1 + rng.normal(0, 0.4, 400)).astype("float32")
X_train, X_val, y_train, y_val = train_test_split(
    X, y_data, test_size=0.25, random_state=42
)
# Tu código aquí.

**Mi interpretación:** escribe aquí.

## 5. Solución de referencia

In [ ]:
def entrenar(nombre):
    keras.utils.set_random_seed(7)
    modelo = keras.Sequential([keras.Input(shape=(1,)), keras.layers.Dense(1)])
    optimizador = (
        keras.optimizers.SGD(0.03) if nombre == "SGD" else keras.optimizers.Adam(0.03)
    )
    modelo.compile(optimizer=optimizador, loss="mse", metrics=["mae"])
    historia = modelo.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=40,
        batch_size=32,
        shuffle=False,
        verbose=0,
    )
    return historia


historias = {nombre: entrenar(nombre) for nombre in ["SGD", "Adam"]}
for nombre, h in historias.items():
    plt.plot(h.history["loss"], label=nombre + " entrenamiento", alpha=0.65)
    plt.plot(h.history["val_loss"], "--", label=nombre + " validación")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
for nombre, h in historias.items():
    print(nombre, "MSE val final:", round(h.history["val_loss"][-1], 4))

Los optimizadores pueden converger a ritmos diferentes. Una sola comparación no establece superioridad general: la tasa, el lote, la escala y los datos también influyen. La semilla y `shuffle=False` ayudan a mantener comparables las condiciones.

## 6. Desafío opcional

Cambia solo la tasa de SGD a 0,001 y después a 0,3. ¿Cambia la velocidad o la estabilidad? Repite desde cero y compara curvas; documenta la configuración.